In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sys

: 

In [ ]:
def plot_ITI(
    sigma,
    delta,
    tau,
    center=None,
    span=None,
    xlim=None,
    ylim=None,
    figsize=(9, 9),
    simplified=False
):
    s_min, s_max = sigma
    d_min, d_max = delta
    t_min, t_max = tau

    fig, ax = plt.subplots(figsize=figsize)

    # --------------------------------------------------
    # Determine viewing window
    # --------------------------------------------------

    if xlim is not None and ylim is not None:

        plot_min_x, plot_max_x = xlim
        plot_min_y, plot_max_y = ylim

    else:

        if center is None:
            center_x = (s_min + s_max) / 2
            center_y = (t_min + t_max) / 2
        else:
            center_x, center_y = center

        if span is None:
            width = s_max - s_min
            height = t_max - t_min
            span_x = span_y = max(width, height) * 2.5

        elif isinstance(span, (int, float)):
            span_x = span_y = span

        else:
            span_x, span_y = span

        plot_min_x = center_x - span_x / 2
        plot_max_x = center_x + span_x / 2

        plot_min_y = center_y - span_y / 2
        plot_max_y = center_y + span_y / 2

    view_width = plot_max_x - plot_min_x
    view_height = plot_max_y - plot_min_y

    x_range = np.linspace(plot_min_x, plot_max_x, 200)

    # --------------------------------------------------
    # D strip
    # --------------------------------------------------

    ax.fill_between(
        x_range,
        x_range + d_min,
        x_range + d_max,
        color="gray",
        alpha=0.15,
        label=r"$D$"
    )

    ax.plot(
        x_range,
        x_range + d_min,
        "k--",
        alpha=0.3,
        linewidth=1
    )

    ax.plot(
        x_range,
        x_range + d_max,
        "k--",
        alpha=0.3,
        linewidth=1
    )

    # --------------------------------------------------
    # V and H intervals
    # --------------------------------------------------

    ax.hlines(
        plot_min_y + 0.05 * view_height,
        s_min,
        s_max,
        color="red",
        linewidth=5,
        label=r"$V$"
    )

    ax.vlines(
        plot_min_x + 0.05 * view_width,
        t_min,
        t_max,
        color="green",
        linewidth=5,
        label=r"$H$"
    )

    # --------------------------------------------------
    # V ∩ H rectangle
    # --------------------------------------------------

    rect = plt.Rectangle(
        (s_min, t_min),
        s_max - s_min,
        t_max - t_min,
        linewidth=2,
        edgecolor="blue",
        facecolor="blue",
        alpha=0.1,
        label=r"$V \cap H$"
    )

    ax.add_patch(rect)

    # --------------------------------------------------
    # V ∩ D ∩ H region
    # --------------------------------------------------

    x_fill = np.linspace(s_min, s_max, 500)

    y_lower = np.maximum(t_min, x_fill + d_min)
    y_upper = np.minimum(t_max, x_fill + d_max)

    ax.fill_between(
        x_fill,
        y_lower,
        y_upper,
        where=(y_upper >= y_lower),
        color="darkviolet",
        alpha=0.5,
        label=r"$V \cap D \cap H$"
    )

    # --------------------------------------------------
    # Formatting
    # --------------------------------------------------

    ax.set_aspect("equal")

    ax.set_xlim(plot_min_x, plot_max_x)
    ax.set_ylim(plot_min_y, plot_max_y)

    ax.grid(True, linestyle=":", alpha=0.6)

    if simplified:
        ax.tick_params(axis="both", which="major", labelsize=20)
    else:
        ax.set_xlabel("s", fontsize=12)
        ax.set_ylabel("t", fontsize=12)
        ax.legend(loc="lower right")

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_R(
    R,
    center=None,
    span=None,
    xlim=None,
    ylim=None,
    figsize=(9, 9),
    connect=False,
    simplified=False
):

    if len(R) == 0:
        raise ValueError("R must contain at least one point.")

    # Sort points by x-coordinate
    R_sorted = sorted(R, key=lambda p: p[0])

    s_vals = np.array([s for s, t in R_sorted])
    t_vals = np.array([t for s, t in R_sorted])

    fig, ax = plt.subplots(figsize=figsize)

    # --------------------------------------------------
    # Determine viewing window
    # --------------------------------------------------

    if xlim is not None and ylim is not None:

        plot_min_x, plot_max_x = xlim
        plot_min_y, plot_max_y = ylim

    else:

        if center is None:
            center_x = (s_vals.min() + s_vals.max()) / 2
            center_y = (t_vals.min() + t_vals.max()) / 2
        else:
            center_x, center_y = center

        if span is None:
            width = max(1e-9, s_vals.max() - s_vals.min())
            height = max(1e-9, t_vals.max() - t_vals.min())

            span_x = span_y = max(width, height) * 1.5

        elif isinstance(span, (int, float)):
            span_x = span_y = span

        else:
            span_x, span_y = span

        plot_min_x = center_x - span_x / 2
        plot_max_x = center_x + span_x / 2

        plot_min_y = center_y - span_y / 2
        plot_max_y = center_y + span_y / 2

    # --------------------------------------------------
    # Dashed path
    # --------------------------------------------------

    if connect and len(R_sorted) > 1:
        ax.plot(
            s_vals,
            t_vals,
            linestyle="--",
            color="black",
            linewidth=1,
            alpha=0.5,
            zorder=1
        )

    # --------------------------------------------------
    # Relation points
    # --------------------------------------------------

    ax.scatter(
        s_vals,
        t_vals,
        color="darkviolet",
        s=100,
        zorder=2,
        label=r"$R$"
    )

    # --------------------------------------------------
    # Formatting
    # --------------------------------------------------

    ax.set_aspect("equal")

    ax.set_xlim(plot_min_x, plot_max_x)
    ax.set_ylim(plot_min_y, plot_max_y)

    ax.grid(True, linestyle=":", alpha=0.6)

    if simplified:
        ax.tick_params(axis="both", which="major", labelsize=20)
        ax.set_xlabel("s", fontsize=25)
        ax.set_ylabel("t", fontsize=25)
    else:
        ax.set_xlabel("s", fontsize=18)
        ax.set_ylabel("t", fontsize=18)
        ax.legend(loc="lower right")

    plt.grid(False)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_ITI(
    sigma=(4, 6), 
    delta=(-sys.maxsize, sys.maxsize), 
    tau=(-sys.maxsize, sys.maxsize),
    # center=(103, 103),
    span=10,
    simplified=True
)

In [ ]:
plot_ITI(
    sigma=(4, 6), 
    delta=(1, 2), 
    tau=(12, 14),
    # center=(103, 103),
    # span=100,
    simplified=True
)

In [ ]:
plot_ITI(
    sigma=(104, 106), 
    delta=(-5, -3), 
    tau=(100, 102),
    center=(103, 103),
    span=10,
    simplified=True
)

In [ ]:
plot_R({(10, 10), (11, 11), (12, 12)}, span=7, center=(11,11), simplified=True, connect=False)

In [ ]:
plot_R({(10, 10), (10, 9), (10, 8), (11, 11), (11, 10), (11, 9), (12, 12), (12, 11), (12, 10)}, span=7, center=(11, 11), simplified=True, connect=False)

In [ ]:
plot_R({(10, 11), (10, 12), (11, 12), (11, 13), (12, 13), (12, 14)}, span=7, center=(11, 11), simplified=True, connect=False)